In [8]:
# ตั้งค่า path และตรวจสอบไฟล์
import os
import glob

BASE_DIR = '/content/drive/MyDrive/OSR'
IMAGE_DIR = os.path.join(BASE_DIR, 'images')
TEMPLATE_PATH = os.path.join(BASE_DIR, 'submission_template.csv')

# ตรวจสอบว่าเจอไฟล์ครบมั้ย
image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, '*.png')))
print(f"พบรูปทั้งหมด: {len(image_files)} ไฟล์")
print(f"ตัวอย่าง 5 ไฟล์แรก:")
for f in image_files[:5]:
    print(f"  {os.path.basename(f)}")

พบรูปทั้งหมด: 847 ไฟล์
ตัวอย่าง 5 ไฟล์แรก:
  constituency_10_1.png
  constituency_10_10.png
  constituency_10_10_page2.png
  constituency_10_11.png
  constituency_10_11_page2.png


In [9]:
# Load template และ group ตาม doc_id
import pandas as pd

df = pd.read_csv(TEMPLATE_PATH)
print(f"Template shape: {df.shape}")
print(df.head())

Template shape: (10053, 5)
                    id             doc_id  row_num    party_name  votes
0  constituency_10_1_1  constituency_10_1        1  ประชาธิปัตย์      0
1  constituency_10_1_2  constituency_10_1        2     ภูมิใจไทย      0
2  constituency_10_1_3  constituency_10_1        3      เศรษฐกิจ      0
3  constituency_10_1_4  constituency_10_1        4      กล้าธรรม      0
4  constituency_10_1_5  constituency_10_1        5         พลวัต      0


In [13]:
# จับคู่แต่ละ doc_id กับไฟล์รูปทุกหน้า
from collections import defaultdict

# สร้าง dict: doc_id -> list of image paths (เรียงตามหน้า)
doc_images = defaultdict(list)

for img_path in image_files:
    filename = os.path.basename(img_path)
    name = filename.replace('.png', '')

    # ตัด _page2, _page3, ... ออก เพื่อให้ได้ doc_id
    if '_page' in name:
        doc_id = name[:name.rfind('_page')]
    else:
        doc_id = name

    doc_images[doc_id].append(img_path)

# เรียงหน้าให้ถูกต้อง (page1 ก่อน)
for doc_id in doc_images:
    doc_images[doc_id].sort()

print(f"จำนวน documents ทั้งหมด: {len(doc_images)}")
print(f"\nตัวอย่าง:")
for doc_id, pages in list(doc_images.items())[:3]:
    print(f"  {doc_id}: {[os.path.basename(p) for p in pages]}")

จำนวน documents ทั้งหมด: 300

ตัวอย่าง:
  constituency_10_1: ['constituency_10_1.png', 'constituency_10_1_page2.png', 'constituency_10_1_page3.png']
  constituency_10_10: ['constituency_10_10.png', 'constituency_10_10_page2.png']
  constituency_10_11: ['constituency_10_11.png', 'constituency_10_11_page2.png', 'constituency_10_11_page3.png']


In [10]:
# import และตั้งค่า Gemini
from google import genai
from google.colab import userdata
from PIL import Image
import json, re, time, os
import pandas as pd
from collections import defaultdict

client = genai.Client(api_key=userdata.get('GEMINI_2'))

In [15]:
# prompt และฟังก์ชัน OCR ด้วย Gemini
from google import genai
from google.genai import types
from PIL import Image
import json, re, time

PROMPT = """คุณคือผู้เชี่ยวชาญอ่านเอกสารผลการเลือกตั้งไทย แบบฟอร์ม สส.6/1

## ขั้นตอนการอ่าน

### ระบุประเภทเอกสาร
- แบบแบ่งเขต (constituency): มีคอลัมน์ "ชื่อ-สกุลผู้สมัคร" และ "สังกัดพรรคการเมือง"
- แบบบัญชีรายชื่อ (party_list): มีคอลัมน์ "ชื่อพรรค" ไม่มีชื่อบุคคล

### วิธีนับแถว (สำคัญมาก)
- นับแถวข้อมูลทั้งหมดในตาราง รวมถึงแถวแรกสุดที่อยู่ติดกับ header
- แถวแรกของข้อมูล **ไม่ใช่ header** — header คือแถวที่มีคำว่า "หมายเลข", "ชื่อ", "สังกัด", "ได้คะแนน"
- ถ้าตารางมี N แถวข้อมูล (ไม่นับ header และไม่นับแถวรวม) ต้องได้ JSON N rows พอดี
- นับจำนวนแถวข้อมูลให้ครบก่อนเริ่มอ่านคะแนน

### วิธีอ่านคะแนน (ทำตามลำดับนี้เท่านั้น)
1. มองหาคอลัมน์ "ได้คะแนน" — คอลัมน์สุดท้ายก่อน "หมายเหตุ"
2. แต่ละแถวจะมีรูปแบบ: ตัวเลขไทย (ตัวหนังสือภาษาไทยในวงเล็บ)
   ตัวอย่าง: ๓๔,๑๗๗ (สามหมื่นสี่พันหนึ่งร้อยเจ็ดสิบเจ็ด) = 34177
3. **อ่านตัวหนังสือในวงเล็บก่อนเสมอ** เพราะแม่นกว่า
4. แปลงเลขไทยในวงเล็บเป็นเลขอารบิก:
   - หน่วย: หนึ่ง=1, สอง=2, สาม=3, สี่=4, ห้า=5, หก=6, เจ็ด=7, แปด=8, เก้า=9
   - ตำแหน่ง: สิบ=×10, ร้อย=×100, พัน=×1000, หมื่น=×10000, แสน=×100000
   - ตัวอย่าง: "หนึ่งหมื่นสี่พันแปดร้อยยี่สิบสาม" = 14823
   - ตัวอย่าง: "สามหมื่นสี่พันหนึ่งร้อยเจ็ดสิบเจ็ด" = 34177
5. ถ้าไม่มีวงเล็บ ให้อ่านตัวเลขไทยโดยตรง: ๑=1, ๒=2, ๓=3, ๔=4, ๕=5, ๖=6, ๗=7, ๘=8, ๙=9, ๐=0

### วิธีอ่านชื่อพรรค
- **constituency**: อ่านจากคอลัมน์ "สังกัดพรรคการเมือง" เท่านั้น ห้ามใช้ชื่อผู้สมัคร
- **party_list**: อ่านจากคอลัมน์ "ชื่อพรรค"
- **คัดลอกชื่อพรรคตามที่เขียนในเอกสารทุกตัวอักษร อย่าแก้ไข อย่าเติม อย่าตัดทอน**

### แถวที่ต้องข้าม
- แถว "รวมคะแนนทั้งสิ้น" — ข้ามทุกกรณี
- แถว header (หมายเลข / ชื่อ / สังกัด / ได้คะแนน) — ข้าม

## รูปแบบ JSON ที่ต้องตอบ
ตอบเป็น JSON เท่านั้น ห้ามมี markdown, code block, หรือข้อความอื่น:

{"type": "constituency", "rows": [{"party_name": "ชื่อพรรคจากคอลัมน์สังกัด", "votes": 1234}, ...]}
{"type": "party_list", "rows": [{"party_name": "ชื่อพรรค", "votes": 1234}, ...]}

## ตรวจสอบก่อนตอบ
- จำนวน rows ใน JSON ต้องเท่ากับจำนวนแถวข้อมูลที่นับได้ (ห้ามขาด ห้ามเกิน)
- แถวแรกของข้อมูลต้องอยู่ใน JSON ด้วย — ห้าม skip
- votes ต้องเป็นตัวเลขอารบิก (integer) เท่านั้น

### การ cross-check คะแนน (บังคับทำทุกแถว)
สำหรับแต่ละแถว ให้ทำดังนี้:
1. อ่านตัวหนังสือในวงเล็บ → แปลงเป็นตัวเลข → ได้ค่า A
2. อ่านตัวเลขไทยหน้าวงเล็บ → แปลงเป็นตัวเลข → ได้ค่า B
3. ถ้า A == B → ใช้ค่านั้น
4. ถ้า A != B → ให้ตรวจสอบภาพซ้ำอีกครั้ง แล้วเลือกค่าที่น่าเชื่อถือกว่า

ตัวอย่างการแปลงตัวเลขไทย:
๑=1, ๒=2, ๓=3, ๔=4, ๕=5, ๖=6, ๗=7, ๘=8, ๙=9, ๐=0
๓๔,๑๗๗ = 34177
๑๙,๓๗๘ = 19378
"""

schema = {
    "type": "object",
    "properties": {
        "type": {"type": "string"},
        "rows": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "party_name": {"type": "string"},
                    "votes": {"type": "integer"}
                },
                "required": ["party_name", "votes"]
            }
        }
    },
    "required": ["type", "rows"]
}

def process_doc_gemini(doc_id, page_paths, max_retries=3):
    """OCR ด้วย Gemini ส่งทุกหน้าพร้อมกัน"""
    images = [Image.open(p) for p in page_paths]

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                # model='gemini-3-flash-lite-preview',
                model='gemini-3-flash-preview',
                contents=[PROMPT] + images,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema
                )
            )
            data = json.loads(response.text)
            return data.get('rows', [])
        except json.JSONDecodeError:
            print(f"  ⚠️ JSON parse error attempt {attempt+1}")
            time.sleep(5)
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower() or 'quota' in err.lower():
                if 'day' in err.lower() or 'RESOURCE_EXHAUSTED' in err:
                    print(f"\n🚨 RPD หมดแล้ว! หยุด — รอ reset ตอนเที่ยงคืน Pacific Time")
                    raise SystemExit("Daily quota exhausted")
                wait = (2 ** attempt) * 15
                print(f"  ⏳ Rate limit, รอ {wait}s (attempt {attempt+1})...")
                time.sleep(wait)
            else:
                print(f"  ❌ {e}")
                return []
    return []

In [101]:
# ใช้แค่ตอนทดสอบ Gemini เท่านั้น จะ test ค่อย uncomment
# ทดสอบกับ 1 document ก่อน
# test_rows = process_doc_gemini('constituency_10_1', doc_images['constituency_10_1'])
# print(f"จำนวน rows: {len(test_rows)}")
# for r in test_rows:
#     print(r)

จำนวน rows: 18
{'party_name': 'ประชาชน', 'votes': 34167}
{'party_name': 'ประชาธิปัตย์', 'votes': 14813}
{'party_name': 'ภูมิใจไทย', 'votes': 14368}
{'party_name': 'เพื่อไทย', 'votes': 6030}
{'party_name': 'รวมไทยสร้างชาติ', 'votes': 2795}
{'party_name': 'โอกาสใหม่', 'votes': 1133}
{'party_name': 'ไทยภักดี', 'votes': 1023}
{'party_name': 'เศรษฐกิจ', 'votes': 979}
{'party_name': 'ไทยสร้างไทย', 'votes': 629}
{'party_name': 'ไทยก้าวใหม่', 'votes': 489}
{'party_name': 'พลวัต', 'votes': 351}
{'party_name': 'กล้าธรรม', 'votes': 244}
{'party_name': 'ปวงชนไทย', 'votes': 168}
{'party_name': 'รักชาติ', 'votes': 165}
{'party_name': 'ทางเลือกใหม่', 'votes': 154}
{'party_name': 'วิชั่นใหม่', 'votes': 113}
{'party_name': 'ประชาธิปไตยใหม่', 'votes': 94}
{'party_name': 'พลังประชารัฐ', 'votes': 80}


In [103]:
# รัน OCR ทุก doc — มี resume + rate limit protection
import os, glob, json, time, re, random
from pathlib import Path
from PIL import Image

RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

doc_ids = sorted(doc_images.keys())
total = len(doc_ids)
failed_docs = []

print(f"เริ่มประมวลผล {total} docs → บันทึกที่ {RESULTS_DIR}")
print("=" * 50)

for i, doc_id in enumerate(doc_ids):
    result_path = os.path.join(RESULTS_DIR, f'{doc_id}.json')

    # Resume: ข้ามถ้ามีไฟล์แล้ว
    if os.path.exists(result_path):
        print(f'[{i+1}/{total}] ⏭️  skip {doc_id}')
        continue

    pages = doc_images[doc_id]
    print(f'[{i+1}/{total}] 🔄 {doc_id} ({len(pages)} หน้า)...', end=' ', flush=True)

    rows = process_doc_gemini(doc_id, pages)

    if rows:
        with open(result_path, 'w', encoding='utf-8') as f:
            json.dump({
                'doc_id': doc_id,
                'num_rows': len(rows),
                'rows': rows
            }, f, ensure_ascii=False, indent=2)
        print(f'✅ {len(rows)} rows')
    else:
        failed_docs.append(doc_id)
        print(f'❌ failed')

    # Rate limit protection: ~4 req/min (ปลอดภัยสำหรับ preview model) >> ปรับใหม่ทีหลังเพราะจ่ายเงินแล้ว
    time.sleep(1)

print("\n" + "=" * 50)
done = total - len(failed_docs) - sum(1 for d in doc_ids if os.path.exists(os.path.join(RESULTS_DIR, f'{d}.json')) and d not in failed_docs)
result_count = len(glob.glob(os.path.join(RESULTS_DIR, '*.json')))
print(f"✅ มี result แล้ว: {result_count}/{total} docs")
print(f"❌ failed: {len(failed_docs)} docs")
if failed_docs:
    print("Failed:", failed_docs)


เริ่มประมวลผล 300 docs → บันทึกที่ /content/drive/MyDrive/OSR/results
[1/300] 🔄 constituency_10_1 (3 หน้า)... ✅ 18 rows
[2/300] 🔄 constituency_10_10 (2 หน้า)... ✅ 16 rows
[3/300] 🔄 constituency_10_11 (3 หน้า)... ✅ 18 rows
[4/300] 🔄 constituency_10_12 (3 หน้า)... ✅ 16 rows
[5/300] 🔄 constituency_10_13 (2 หน้า)... ✅ 15 rows
[6/300] 🔄 constituency_10_14 (3 หน้า)... ✅ 16 rows
[7/300] 🔄 constituency_10_16 (2 หน้า)... ✅ 15 rows
[8/300] 🔄 constituency_10_17 (2 หน้า)... ✅ 15 rows
[9/300] 🔄 constituency_10_18 (3 หน้า)... ✅ 18 rows
[10/300] 🔄 constituency_10_19 (3 หน้า)... ✅ 14 rows
[11/300] 🔄 constituency_10_2 (3 หน้า)... ✅ 15 rows
[12/300] 🔄 constituency_10_20 (2 หน้า)... ✅ 14 rows
[13/300] 🔄 constituency_10_21 (2 หน้า)... ✅ 15 rows
[14/300] 🔄 constituency_10_22 (2 หน้า)... ✅ 14 rows
[15/300] 🔄 constituency_10_23 (3 หน้า)... ✅ 18 rows
[16/300] 🔄 constituency_10_24 (3 หน้า)... ✅ 13 rows
[17/300] 🔄 constituency_10_25 (2 หน้า)... ✅ 14 rows
[18/300] 🔄 constituency_10_26 (2 หน้า)... ✅ 14 rows
[19/3

In [108]:
# เปรียบเทียบ 2 รอบ และหา docs ที่ต้องรัน re-run ด้วย model แรงกว่า
# flash lite รันไปก่อนหน้า flash โดยใช้ script เดี๋ยว แต่ไปเก็บ folder แยก
import glob, json, os
import pandas as pd

RESULTS_DIR_1 = os.path.join(BASE_DIR, 'results')        # รอบ 1
RESULTS_DIR_2 = os.path.join(BASE_DIR, 'results_flash_lite') # รอบ 2

MISMATCH_THRESHOLD = 0.3

result_files_1 = sorted(glob.glob(os.path.join(RESULTS_DIR_1, '*.json')))

summary = []
rerun_docs = []

for fpath1 in result_files_1:
    doc_id = os.path.basename(fpath1).replace('.json', '')
    fpath2 = os.path.join(RESULTS_DIR_2, f'{doc_id}.json')

    # ข้ามถ้าไม่มีรอบ 2
    if not os.path.exists(fpath2):
        continue

    with open(fpath1, encoding='utf-8') as f:
        rows1 = json.load(f)['rows']
    with open(fpath2, encoding='utf-8') as f:
        rows2 = json.load(f)['rows']

    total = min(len(rows1), len(rows2))
    if total == 0:
        continue

    mismatches = sum(1 for r1, r2 in zip(rows1, rows2) if r1['votes'] != r2['votes'])
    mismatch_ratio = mismatches / total

    summary.append({
        'doc_id': doc_id,
        'total_rows': total,
        'mismatches': mismatches,
        'mismatch_ratio': round(mismatch_ratio, 3),
        'need_rerun': mismatch_ratio > MISMATCH_THRESHOLD
    })

    if mismatch_ratio > MISMATCH_THRESHOLD:
        rerun_docs.append(doc_id)

df = pd.DataFrame(summary)

# สรุป
print(f"=== Ensemble Summary (threshold={MISMATCH_THRESHOLD*100:.0f}%) ===")
print(f"docs ทั้งหมด:       {len(df)}")
print(f"ตรงกันดี:          {(~df['need_rerun']).sum()} docs")
print(f"ต้อง re-run:        {df['need_rerun'].sum()} docs")
print(f"\n=== docs ที่ต้อง re-run ===")
print(df[df['need_rerun']][['doc_id','total_rows','mismatches','mismatch_ratio']].to_string(index=False))

# บันทึกรายชื่อ docs ที่ต้อง re-run
rerun_path = os.path.join(BASE_DIR, 'rerun_docs.json')
with open(rerun_path, 'w', encoding='utf-8') as f:
    json.dump(rerun_docs, f, ensure_ascii=False, indent=2)
print(f"\n✅ บันทึก rerun list → {rerun_path}")

=== Ensemble Summary (threshold=30%) ===
docs ทั้งหมด:       300
ตรงกันดี:          237 docs
ต้อง re-run:        63 docs

=== docs ที่ต้อง re-run ===
            doc_id  total_rows  mismatches  mismatch_ratio
constituency_10_10          16           5           0.312
constituency_10_12          16           7           0.438
constituency_10_13          15           6           0.400
constituency_10_14          16           5           0.312
constituency_10_21          15           6           0.400
constituency_10_22          14           6           0.429
constituency_10_25          14           6           0.429
constituency_10_27          12           4           0.333
constituency_10_33          14           5           0.357
 constituency_10_4          13           4           0.308
 constituency_10_5          16           5           0.312
 constituency_10_6          16           9           0.562
 constituency_10_8          16           5           0.312
 constituency_11_1      

In [109]:
# Re-run เฉพาะ docs ที่ไม่ผ่าน threshold ด้วย Gemini 3 Pro
RESULTS_DIR_3 = os.path.join(BASE_DIR, 'results_round3_pro')
os.makedirs(RESULTS_DIR_3, exist_ok=True)

with open(rerun_path, encoding='utf-8') as f:
    rerun_docs = json.load(f)

print(f"Re-run {len(rerun_docs)} docs ด้วย gemini-3.1-pro-preview")
print(f"RPD limit ~20 — ใช้ได้ {min(len(rerun_docs), 20)} docs/วัน") # จ่ายตังละจ้า

for i, doc_id in enumerate(rerun_docs):
    result_path = os.path.join(RESULTS_DIR_3, f'{doc_id}.json')
    if os.path.exists(result_path):
        print(f"[{i+1}/{len(rerun_docs)}] ⏭️  skip {doc_id}")
        continue

    pages = doc_images[doc_id]
    print(f"[{i+1}/{len(rerun_docs)}] 🔄 {doc_id}...", end=' ', flush=True)

    images = [Image.open(p) for p in pages]
    try:
        response = client.models.generate_content(
            model='gemini-3.1-pro-preview',
            contents=[PROMPT] + images,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=schema
            )
        )
        rows = json.loads(response.text).get('rows', [])
        if rows:
            with open(result_path, 'w', encoding='utf-8') as f:
                json.dump({'doc_id': doc_id, 'num_rows': len(rows), 'rows': rows},
                          f, ensure_ascii=False, indent=2)
            print(f"✅ {len(rows)} rows")
        else:
            print("❌ empty")
    except Exception as e:
        err = str(e)
        if 'day' in err.lower() or 'RESOURCE_EXHAUSTED' in err:
            print(f"\n🚨 RPD หมดแล้ว! รอ reset บ่าย 3 โมงเวลาไทย")
            break
        print(f"❌ {e}")

    time.sleep(1)

print(f"\n✅ เสร็จแล้ว {len(os.listdir(RESULTS_DIR_3))} docs")

Re-run 63 docs ด้วย gemini-3.1-pro-preview
RPD limit ~20 — ใช้ได้ 20 docs/วัน
[1/63] 🔄 constituency_10_10... ✅ 16 rows
[2/63] 🔄 constituency_10_12... ✅ 16 rows
[3/63] 🔄 constituency_10_13... ✅ 15 rows
[4/63] 🔄 constituency_10_14... ✅ 16 rows
[5/63] 🔄 constituency_10_21... ✅ 15 rows
[6/63] 🔄 constituency_10_22... ✅ 14 rows
[7/63] 🔄 constituency_10_25... ✅ 14 rows
[8/63] 🔄 constituency_10_27... ✅ 12 rows
[9/63] 🔄 constituency_10_33... ✅ 14 rows
[10/63] 🔄 constituency_10_4... ✅ 13 rows
[11/63] 🔄 constituency_10_5... ✅ 16 rows
[12/63] 🔄 constituency_10_6... ✅ 16 rows
[13/63] 🔄 constituency_10_8... ✅ 16 rows
[14/63] 🔄 constituency_11_1... ✅ 8 rows
[15/63] 🔄 constituency_11_4... ✅ 11 rows
[16/63] 🔄 constituency_12_6... ✅ 8 rows
[17/63] 🔄 constituency_12_7... ✅ 9 rows
[18/63] 🔄 constituency_12_8... ✅ 11 rows
[19/63] 🔄 constituency_13_8... ✅ 8 rows
[20/63] 🔄 constituency_14_4... ✅ 8 rows
[21/63] 🔄 constituency_16_1... ✅ 11 rows
[22/63] 🔄 constituency_16_3... ✅ 9 rows
[23/63] 🔄 constituency_16_

In [118]:
# เช็ค total votes สำหรับ docs ที่ใช้ Flash เท่านั้น
import json, os, time
from PIL import Image

RESULTS_FLASH = os.path.join(BASE_DIR, 'results_flash')

PROMPT_TOTAL = """ดูแถวสุดท้ายของตารางที่มีคำว่า "รวมคะแนนทั้งสิ้น"
อ่านตัวเลขในวงเล็บของแถวนั้น แปลงเป็นเลขอารบิก
ตอบเป็น JSON เท่านั้น ไม่มี markdown:
{"total": 12345}"""

# โหลด rerun list ที่ส่งไป Pro แล้ว
with open(os.path.join(BASE_DIR, 'rerun_docs.json'), encoding='utf-8') as f:
    rerun_docs = set(json.load(f))

# หา docs ที่ใช้ Flash เท่านั้น
flash_only_docs = [
    doc_id for doc_id in sorted(doc_images.keys())
    if doc_id not in rerun_docs
]
print(f"Flash-only docs: {len(flash_only_docs)}")

# อ่าน total จากภาพ แล้วเทียบกับ sum
mismatch_total = []
results = []

for i, doc_id in enumerate(flash_only_docs):
    # โหลด rows ที่มีอยู่
    fpath = os.path.join(RESULTS_FLASH, f'{doc_id}.json')
    if not os.path.exists(fpath):
        continue
    with open(fpath, encoding='utf-8') as f:
        ocr_rows = json.load(f)['rows']
    our_sum = sum(r['votes'] for r in ocr_rows)

    # อ่าน total จากภาพด้วย Gemini
    pages = doc_images[doc_id]
    images = [Image.open(p) for p in pages]

    print(f"[{i+1}/{len(flash_only_docs)}] {doc_id}...", end=' ', flush=True)

    try:
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=[PROMPT_TOTAL] + images,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema={
                    "type": "object",
                    "properties": {"total": {"type": "integer"}},
                    "required": ["total"]
                }
            )
        )
        image_total = json.loads(response.text).get('total', 0)
        match = our_sum == image_total

        results.append({
            'doc_id': doc_id,
            'our_sum': our_sum,
            'image_total': image_total,
            'diff': abs(our_sum - image_total),
            'match': match
        })

        if not match:
            mismatch_total.append(doc_id)
            print(f"❌ our={our_sum:,} image={image_total:,} diff={abs(our_sum-image_total):,}")
        else:
            print(f"✅ {our_sum:,}")

    except Exception as e:
        print(f"⚠️ {e}")

    time.sleep(0.1)

# สรุป
df_results = pd.DataFrame(results)
print(f"\n=== Total Votes Check ===")
print(f"ตรงกัน:    {df_results['match'].sum()} docs")
print(f"ไม่ตรงกัน: {(~df_results['match']).sum()} docs")

# docs ที่ diff เยอะที่สุด
print(f"\n=== Top 10 ที่ต่างกันมากที่สุด ===")
print(df_results[~df_results['match']].sort_values('diff', ascending=False)
      .head(10)[['doc_id','our_sum','image_total','diff']].to_string(index=False))

# บันทึก flag list
flag_path = os.path.join(BASE_DIR, 'flagged_by_total.json')
with open(flag_path, 'w', encoding='utf-8') as f:
    json.dump(mismatch_total, f, ensure_ascii=False, indent=2)
print(f"\n✅ บันทึก flagged list → {flag_path} ({len(mismatch_total)} docs)")

Flash-only docs: 237
[1/237] constituency_10_1... ❌ our=77,015 image=77,075 diff=60
[2/237] constituency_10_11... ✅ 94,993
[3/237] constituency_10_16... ✅ 94,218
[4/237] constituency_10_17... ✅ 71,720
[5/237] constituency_10_18... ❌ our=80,200 image=80,204 diff=4
[6/237] constituency_10_19... ❌ our=92,121 image=92,191 diff=70
[7/237] constituency_10_2... ✅ 83,874
[8/237] constituency_10_20... ✅ 81,935
[9/237] constituency_10_23... ❌ our=95,233 image=94,233 diff=1,000
[10/237] constituency_10_24... ✅ 89,356
[11/237] constituency_10_26... ✅ 90,037
[12/237] constituency_10_28... ❌ our=91,556 image=91,560 diff=4
[13/237] constituency_10_29... ✅ 96,248
[14/237] constituency_10_3... ✅ 76,775
[15/237] constituency_10_30... ✅ 92,533
[16/237] constituency_10_31... ❌ our=98,541 image=98,491 diff=50
[17/237] constituency_10_32... ❌ our=86,151 image=85,631 diff=520
[18/237] constituency_10_7... ❌ our=85,141 image=85,181 diff=40
[19/237] constituency_10_9... ✅ 95,111
[20/237] constituency_11_2... ✅

In [16]:
# Re-run docs ที่ total ไม่ตรง ด้วย Gemini Pro → บันทึกใน results_pro
import json, os, time
from PIL import Image

RESULTS_PRO = os.path.join(BASE_DIR, 'results_pro')
os.makedirs(RESULTS_PRO, exist_ok=True)

# โหลด flagged list
with open(os.path.join(BASE_DIR, 'flagged_by_total.json'), encoding='utf-8') as f:
    flagged_docs = json.load(f)

print(f"Re-run {len(flagged_docs)} docs ด้วย gemini-3.1-pro-preview")

for i, doc_id in enumerate(flagged_docs):
    result_path = os.path.join(RESULTS_PRO, f'{doc_id}.json')

    # ข้ามถ้ารันไปแล้ว
    if os.path.exists(result_path):
        print(f'[{i+1}/{len(flagged_docs)}] ⏭️  skip {doc_id}')
        continue

    pages = doc_images[doc_id]
    images = [Image.open(p) for p in pages]
    print(f'[{i+1}/{len(flagged_docs)}] 🔄 {doc_id}...', end=' ', flush=True)

    try:
        response = client.models.generate_content(
            model='gemini-3.1-pro-preview',
            contents=[PROMPT] + images,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=schema
            )
        )
        rows = json.loads(response.text).get('rows', [])

        if rows:
            with open(result_path, 'w', encoding='utf-8') as f:
                json.dump({
                    'doc_id': doc_id,
                    'num_rows': len(rows),
                    'rows': rows
                }, f, ensure_ascii=False, indent=2)
            print(f'✅ {len(rows)} rows')
        else:
            print('❌ empty')

    except Exception as e:
        err = str(e)
        if 'day' in err.lower() or 'RESOURCE_EXHAUSTED' in err:
            print(f'\n🚨 RPD หมดแล้ว! รอ reset บ่าย 3 โมงเวลาไทย')
            break
        print(f'❌ {e}')

    time.sleep(2.5)

print(f'\n✅ เสร็จแล้ว {len(os.listdir(RESULTS_PRO))} docs ใน results_pro')

Re-run 118 docs ด้วย gemini-3.1-pro-preview
[1/118] ⏭️  skip constituency_10_1
[2/118] ⏭️  skip constituency_10_18
[3/118] ⏭️  skip constituency_10_19
[4/118] ⏭️  skip constituency_10_23
[5/118] ⏭️  skip constituency_10_28
[6/118] ⏭️  skip constituency_10_31
[7/118] ⏭️  skip constituency_10_32
[8/118] ⏭️  skip constituency_10_7
[9/118] ⏭️  skip constituency_12_5
[10/118] ⏭️  skip constituency_13_6
[11/118] ⏭️  skip constituency_14_3
[12/118] ⏭️  skip constituency_15_2
[13/118] ⏭️  skip constituency_16_2
[14/118] ⏭️  skip constituency_20_5
[15/118] ⏭️  skip constituency_30_13
[16/118] ⏭️  skip constituency_30_2
[17/118] ⏭️  skip constituency_33_4
[18/118] ⏭️  skip party_list_10_10
[19/118] ⏭️  skip party_list_10_11
[20/118] ⏭️  skip party_list_10_13
[21/118] ⏭️  skip party_list_10_14
[22/118] ⏭️  skip party_list_10_16
[23/118] ⏭️  skip party_list_10_18
[24/118] ⏭️  skip party_list_10_19
[25/118] ⏭️  skip party_list_10_20
[26/118] ⏭️  skip party_list_10_21
[27/118] ⏭️  skip party_list_10

## **อันนี้เป็น process ตรวจไฟล์ตอนรันครั้งแรก เพราะยังไม่มั่นใจว่า format ถูกต้องไหม แต่หลังจากรันหลายรอบก็พบว่าไม่ต้องตรวจ step นี้อีก เพราะ format คงที่แล้ว**

In [106]:
# Sanity check — ดูภาพรวมก่อน submit
import pandas as pd
import glob, json, os

result_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json')))
print(f"มี result แล้ว {len(result_files)}/300 docs\n")

summary = []
for fpath in result_files:
    with open(fpath, encoding='utf-8') as f:
        data = json.load(f)
    rows = data['rows']
    votes = [r['votes'] for r in rows]
    doc_type = 'party_list' if 'party_list' in data['doc_id'] else 'constituency'
    summary.append({
        'doc_id': data['doc_id'],
        'type': doc_type,
        'num_rows': len(rows),
        'total_votes': sum(votes),
        'max_votes': max(votes) if votes else 0,
        'min_votes': min(votes) if votes else 0,
        'has_zero': any(v == 0 for v in votes),
    })

df_summary = pd.DataFrame(summary)

# --- ตรวจ row count ผิดปกติ ---
print("=== ❌ Row count ผิดปกติ ===")
pl_wrong = df_summary[(df_summary['type']=='party_list') & (df_summary['num_rows'] != 57)]
con_wrong = df_summary[(df_summary['type']=='constituency') & ((df_summary['num_rows'] < 4) | (df_summary['num_rows'] > 19))]
print(f"party_list ≠ 57 rows: {len(pl_wrong)} docs")
if len(pl_wrong): print(pl_wrong[['doc_id','num_rows']].to_string(index=False))
print(f"constituency ไม่อยู่ในช่วง 4-19 rows: {len(con_wrong)} docs")
if len(con_wrong): print(con_wrong[['doc_id','num_rows']].to_string(index=False))

# --- ตรวจ votes = 0 ---
print("\n=== ⚠️ มี votes = 0 ===")
zeros = df_summary[df_summary['has_zero']]
print(f"{len(zeros)} docs มีแถวที่ votes=0")
if len(zeros): print(zeros[['doc_id','num_rows']].to_string(index=False))

# --- ตรวจ votes สูงผิดปกติ ---
print("\n=== ⚠️ votes สูงผิดปกติ (> 200,000) ===")
high = df_summary[df_summary['max_votes'] > 200000]
print(f"{len(high)} docs")
if len(high): print(high[['doc_id','max_votes']].to_string(index=False))

# --- สถิติรวม ---
print("\n=== 📊 สถิติรวม ===")
print(df_summary.groupby('type')[['num_rows','total_votes','max_votes']].describe().round(0).to_string())


มี result แล้ว 300/300 docs

=== ❌ Row count ผิดปกติ ===
party_list ≠ 57 rows: 0 docs
constituency ไม่อยู่ในช่วง 4-19 rows: 0 docs

=== ⚠️ มี votes = 0 ===
3 docs มีแถวที่ votes=0
            doc_id  num_rows
 constituency_14_2        18
constituency_30_13         8
 constituency_30_5         9

=== ⚠️ votes สูงผิดปกติ (> 200,000) ===
0 docs

=== 📊 สถิติรวม ===
             num_rows                                          total_votes                                                                max_votes                                                              
                count  mean  std   min   25%   50%   75%   max       count     mean     std      min      25%      50%      75%       max     count     mean     std      min      25%      50%      75%      max
type                                                                                                                                                                                                             
consti

In [93]:
# เช็ค 3 docs ที่มี votes=0
for doc_id in ['constituency_14_2', 'constituency_30_13', 'constituency_30_5']:
    fpath = os.path.join(RESULTS_DIR, f'{doc_id}.json')
    with open(fpath, encoding='utf-8') as f:
        data = json.load(f)
    print(f"\n=== {doc_id} ===")
    for r in data['rows']:
        marker = " ← ZERO" if r['votes'] == 0 else ""
        print(f"  {r['party_name']:25s} {r['votes']:>8,}{marker}")


=== constituency_14_2 ===
  ภูมิใจไทย                   53,374
  ประชาชน                     25,836
  เพื่อไทย                     9,030
  เศรษฐกิจ                     1,953
  รวมไทยสร้างชาติ              1,356
  ประชาธิปัตย์                 1,143
  ไทยก้าวใหม่                    699
  กล้าธรรม                       662
  พลังประชารัฐ                   244
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO
                                   0 ← ZERO

=== constituency_30_13 ===
  เพื่อไทย                    28,157
  ประชาชน                     25,404
  ภูมิใจไทย                    5,877
  เศรษฐกิจ                     3,028
  พลังประชารัฐ                 1,944
  ประชาธิปัตย์                 

เช็ค doc ที่รันแล้วมีค่า 0 แล้วพบว่าเอกสารเป็นแบบนั้นจริง

## **ทำไฟล์ submission**

In [17]:
# Build submission.csv — priority: results_pro > results_flash
import pandas as pd
import glob, json, os

RESULTS_PRO   = os.path.join(BASE_DIR, 'results_pro')
RESULTS_FLASH = os.path.join(BASE_DIR, 'results_flash')

df_template = pd.read_csv(TEMPLATE_PATH)
print(f"Template: {len(df_template)} rows")

pro_files   = set(os.path.basename(f) for f in glob.glob(os.path.join(RESULTS_PRO, '*.json')))
flash_files = set(os.path.basename(f) for f in glob.glob(os.path.join(RESULTS_FLASH, '*.json')))

print(f"results_pro:   {len(pro_files)} docs")
print(f"results_flash: {len(flash_files)} docs")

vote_map = {}

all_doc_ids = sorted(set(df_template['doc_id'].unique()))

for doc_id in all_doc_ids:
    fname = f'{doc_id}.json'

    # priority: pro ก่อน ถ้าไม่มีค่อยใช้ flash
    if fname in pro_files:
        fpath = os.path.join(RESULTS_PRO, fname)
        source = 'pro'
    elif fname in flash_files:
        fpath = os.path.join(RESULTS_FLASH, fname)
        source = 'flash'
    else:
        print(f"  ⚠️ ไม่พบ result ของ {doc_id} — ใช้ 0 แทน")
        continue

    with open(fpath, encoding='utf-8') as f:
        ocr_rows = json.load(f)['rows']

    tmpl_rows = df_template[df_template['doc_id'] == doc_id]
    for _, tmpl_row in tmpl_rows.iterrows():
        row_num = tmpl_row['row_num']
        idx = row_num - 1
        votes = ocr_rows[idx]['votes'] if idx < len(ocr_rows) else 0
        vote_map[(doc_id, row_num)] = votes

# สร้าง submission
df_template['votes'] = df_template.apply(
    lambda r: vote_map.get((r['doc_id'], r['row_num']), 0), axis=1
)
submission = df_template[['id', 'votes']].copy()

# สถิติ
pro_doc_ids   = set(f.replace('.json','') for f in pro_files)
flash_doc_ids = set(f.replace('.json','') for f in flash_files)
used_pro   = len([d for d in all_doc_ids if d in pro_doc_ids])
used_flash = len([d for d in all_doc_ids if d not in pro_doc_ids and d in flash_doc_ids])

print(f"\n=== Source Summary ===")
print(f"ใช้ Pro:   {used_pro} docs")
print(f"ใช้ Flash: {used_flash} docs")
print(f"ไม่มีข้อมูล: {len(all_doc_ids) - used_pro - used_flash} docs")

# บันทึก
OUTPUT_PATH = os.path.join(BASE_DIR, 'submission.csv')
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ บันทึก submission.csv → {OUTPUT_PATH}")
print(f"   {len(submission)} rows, votes=0 อยู่ {(submission['votes']==0).sum()} rows")
print(submission.head(10).to_string(index=False))

Template: 10053 rows
results_pro:   181 docs
results_flash: 300 docs

=== Source Summary ===
ใช้ Pro:   181 docs
ใช้ Flash: 119 docs
ไม่มีข้อมูล: 0 docs

✅ บันทึก submission.csv → /content/drive/MyDrive/OSR/submission.csv
   10053 rows, votes=0 อยู่ 14 rows
                  id  votes
 constituency_10_1_1  34167
 constituency_10_1_2  14813
 constituency_10_1_3  14368
 constituency_10_1_4   6030
 constituency_10_1_5   2075
 constituency_10_1_6   1133
 constituency_10_1_7   1023
 constituency_10_1_8    979
 constituency_10_1_9    629
constituency_10_1_10    489


In [107]:
#script เก่าที่ใช้ก่อนมีการรวมผลจาก 2 model
# Build submission.csv จาก results
# import pandas as pd
# import glob, json, os

# # โหลด template
# df_template = pd.read_csv(TEMPLATE_PATH)
# print(f"Template: {len(df_template)} rows")

# # โหลด results ทั้งหมด
# result_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json')))
# print(f"Results: {len(result_files)} docs")

# # สร้าง vote_map: (doc_id, row_num) -> votes
# vote_map = {}

# for fpath in result_files:
#     with open(fpath, encoding='utf-8') as f:
#         data = json.load(f)

#     doc_id = data['doc_id']
#     ocr_rows = data['rows']

#     tmpl_rows = df_template[df_template['doc_id'] == doc_id].reset_index(drop=True)

#     for _, tmpl_row in tmpl_rows.iterrows():
#         row_num = tmpl_row['row_num']
#         idx = row_num - 1  # row_num เริ่มจาก 1
#         if 0 <= idx < len(ocr_rows):
#             vote_map[(doc_id, row_num)] = ocr_rows[idx]['votes']
#         else:
#             vote_map[(doc_id, row_num)] = 0  # fallback

# # สร้าง submission
# df_template['votes'] = df_template.apply(
#     lambda r: vote_map.get((r['doc_id'], r['row_num']), 0), axis=1
# )
# submission = df_template[['id', 'votes']].copy()

# # บันทึก
# OUTPUT_PATH = os.path.join(BASE_DIR, 'submission.csv')
# submission.to_csv(OUTPUT_PATH, index=False)
# print(f"\n✅ บันทึก submission.csv → {OUTPUT_PATH}")
# print(f"   {len(submission)} rows, votes=0 อยู่ {(submission['votes']==0).sum()} rows")
# print(submission.head(10).to_string(index=False))

Template: 10053 rows
Results: 300 docs

✅ บันทึก submission.csv → /content/drive/MyDrive/OSR/submission.csv
   10053 rows, votes=0 อยู่ 14 rows
                  id  votes
 constituency_10_1_1  34167
 constituency_10_1_2  14813
 constituency_10_1_3  14308
 constituency_10_1_4   6030
 constituency_10_1_5   2075
 constituency_10_1_6   1133
 constituency_10_1_7   1023
 constituency_10_1_8    979
 constituency_10_1_9    629
constituency_10_1_10    489
